In [ ]:
# ======================================================================
# TAAPSHAMAN: one-cell Colab web app
# Paste this whole cell into Google Colab and run it.
# If your training CSV is not already in /content, you will be asked to
# upload taapshaman_synthetic_augmented_dataset.csv.
# ======================================================================
import glob, os
from threading import Thread

import numpy as np
import pandas as pd
from flask import Flask, jsonify, request, Response
from werkzeug.serving import make_server
from google.colab import files, output
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PORT = 5000
TARGETS = [
    "Daily Maximum Indoor Temperature (C, synthetic)",
    "Indoor Temperature During Hot Day (C, synthetic)",
    "Hours Above 28C (synthetic, 0-24)",
]
EXCLUDED = ["Building ID", "Satellite Image Feature"] + TARGETS
SECTIONS = {
    "Building": [
        "Height (m)", "Footprint Area (sq.m)", "Num. of Floors",
        "Distance to City Center (km)", "Construction Material",
        "Building Function", "Region", "Building Density",
        "Zoning Classification", "Land Use Type",
    ],
    "Envelope & ventilation": [
        "Roof Material (synthetic)", "Roof Colour (synthetic)",
        "Wall Material (synthetic)", "Roof Insulation (synthetic, mm)",
        "Window Area Ratio (synthetic, %)", "External Shading (synthetic)",
        "Natural Ventilation (synthetic)", "Air Conditioning (synthetic)",
    ],
    "Weather & surroundings": [
        "Outdoor Temperature Hot Day (C, synthetic)",
        "Relative Humidity Hot Day (%, synthetic)",
        "Solar Radiation (W/m2, synthetic)", "Wind Speed (m/s, synthetic)",
        "Tree Canopy Cover (%, synthetic)",
        "Urban Heat Island Intensity (C, synthetic)",
    ],
}

# ---------------------------------------------------------------- 1. data
def get_csv():
    found = [f for f in glob.glob("/content/*.csv")
             if "augmented" in os.path.basename(f).lower()]
    if not found:
        print("Upload taapshaman_synthetic_augmented_dataset.csv")
        up = files.upload()
        found = [os.path.join("/content", n) for n in up if n.lower().endswith(".csv")]
    if not found:
        raise SystemExit("No CSV uploaded.")
    return max(found, key=os.path.getmtime)

df = pd.read_csv(get_csv())
missing = [c for c in TARGETS if c not in df.columns]
if missing:
    raise SystemExit(f"Missing target columns: {missing}. Use the augmented dataset.")

# --------------------------------------------------------------- 2. models
features = [c for c in df.columns if c not in EXCLUDED]
numeric = [c for c in features if pd.api.types.is_numeric_dtype(df[c])]
categorical = [c for c in features if c not in numeric]

print("Training three models (a few seconds)...")
MODELS, METRICS = {}, {}
for target in TARGETS:
    work = df[features + [target]].copy()
    work[target] = pd.to_numeric(work[target], errors="coerce")
    work = work.dropna(subset=[target])
    X_tr, X_te, y_tr, y_te = train_test_split(
        work[features], work[target], test_size=0.20, random_state=42)
    pre = ColumnTransformer([
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical),
    ])
    model = Pipeline([
        ("preprocessing", pre),
        ("regressor", ExtraTreesRegressor(
            n_estimators=250, min_samples_leaf=2, random_state=42, n_jobs=-1)),
    ])
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    MODELS[target] = model
    METRICS[target] = {"mae": round(mean_absolute_error(y_te, pred), 2),
                       "r2": round(r2_score(y_te, pred), 3)}

# Form schema built from the data, so options always match the training set.
fields = {}
for col in features:
    if col in numeric:
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        fields[col] = {"name": col, "type": "number", "min": float(s.min()),
                       "max": float(s.max()), "default": float(s.median()),
                       "step": 1 if bool((s == s.round()).all()) else 0.1}
    else:
        ch = sorted(df[col].dropna().astype(str).unique().tolist()) or ["Unknown"]
        fields[col] = {"name": col, "type": "choice", "choices": ch, "default": ch[0]}

sections, used = [], set()
for title, cols in SECTIONS.items():
    present = [fields[c] for c in cols if c in fields]
    used.update(f["name"] for f in present)
    if present:
        sections.append({"title": title, "fields": present})
rest = [fields[c] for c in features if c not in used]
if rest:
    sections.append({"title": "Additional features", "fields": rest})
SCHEMA = {"sections": sections, "metrics": METRICS, "n_rows": len(df)}


def suggest(row):
    s = []
    g = lambda k, d=0: row.get(k, d)
    colour = str(g("Roof Colour (synthetic)", "")).lower()
    shading = str(g("External Shading (synthetic)", "")).lower()
    vent = str(g("Natural Ventilation (synthetic)", "")).lower()
    wall = str(g("Wall Material (synthetic)", "")).lower()
    ins = g("Roof Insulation (synthetic, mm)")
    if any(w in colour for w in ("dark", "black")):
        s.append("Evaluate a reflective, light-coloured roof coating.")
    if ins < 50:
        s.append("Assess whether additional roof insulation is suitable.")
    if shading in ("none", "no", "low", "false"):
        s.append("Evaluate external shading for sun-exposed windows.")
    if g("Window Area Ratio (synthetic, %)") > 30:
        s.append("Review window solar gain, glazing and external shading.")
    if vent in ("none", "no", "poor", "low", "false"):
        s.append("Assess cross-ventilation or night ventilation where outdoor conditions permit.")
    if g("Tree Canopy Cover (%, synthetic)") < 20:
        s.append("Consider suitable tree cover and shaded outdoor areas.")
    if g("Outdoor Temperature Hot Day (C, synthetic)") > 30:
        s.append("Prioritise passive cooling and reduction of solar heat gain.")
    if "concrete" in wall and ins < 50:
        s.append("Assess wall heat transfer and possible wall insulation.")
    return s or ["Review the envelope and ventilation using a site-specific "
                 "energy or thermal assessment."]

# ------------------------------------------------------------ 3. web page
HTML = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>TAAPSHAMAN · Thermal Resilience Assessment</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600;9..144,700&family=Inter:wght@400;500;600&display=swap" rel="stylesheet">
<style>
:root{
  --bg:#faf6f0;--card:#fffdf9;--ink:#2a2118;--muted:#7a6c5d;--line:#eadfce;
  --accent:#d9480f;--accent-soft:#fdebdc;--good:#2f9e44;--mid:#e8a317;--bad:#d9342b;
  --shadow:0 1px 2px rgba(60,35,10,.06),0 8px 24px rgba(60,35,10,.06);
}
@media (prefers-color-scheme:dark){
  :root{--bg:#17120d;--card:#211a13;--ink:#f3eadf;--muted:#a69886;--line:#33291f;
  --accent:#ff8a4c;--accent-soft:#3a2415;--good:#51cf66;--mid:#fcc419;--bad:#ff6b61;--shadow:none}
}
*{box-sizing:border-box}
html,body{margin:0}
body{background:var(--bg);color:var(--ink);font:15px/1.5 Inter,system-ui,sans-serif;
  background-image:radial-gradient(900px 400px at 85% -10%,var(--accent-soft),transparent 70%)}
.wrap{max-width:1180px;margin:0 auto;padding:32px 20px 56px}
header{display:flex;align-items:center;gap:14px;margin-bottom:6px}
.logo{width:44px;height:44px;border-radius:12px;background:var(--accent);display:grid;place-items:center;color:#fff}
h1{font:700 30px/1.1 Fraunces,Georgia,serif;margin:0;letter-spacing:-.01em}
.sub{color:var(--muted);margin:6px 0 26px;max-width:680px}
.grid{display:grid;grid-template-columns:minmax(0,1.25fr) minmax(0,1fr);gap:24px;align-items:start}
@media(max-width:900px){.grid{grid-template-columns:1fr}}
.card{background:var(--card);border:1px solid var(--line);border-radius:18px;box-shadow:var(--shadow)}
.sec{padding:20px 22px;margin-bottom:18px}
.sec h2{font:600 18px Fraunces,Georgia,serif;margin:0 0 14px;display:flex;align-items:center;gap:8px}
.sec h2:before{content:"";width:8px;height:8px;border-radius:50%;background:var(--accent)}
.fields{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:16px 22px}
@media(max-width:600px){.fields{grid-template-columns:1fr}}
.f label,.f .lab{display:flex;justify-content:space-between;align-items:baseline;font-weight:500;font-size:13px;margin-bottom:6px;gap:8px}
.f .lab span:first-child{color:var(--muted)}
.val{font-variant-numeric:tabular-nums;font-weight:600;color:var(--ink)}
input[type=range]{width:100%;accent-color:var(--accent);margin:0}
select{width:100%;padding:9px 10px;border-radius:10px;border:1px solid var(--line);background:var(--bg);color:var(--ink);font:inherit}
.pills{display:flex;flex-wrap:wrap;gap:6px}
.pills input{position:absolute;opacity:0;pointer-events:none}
.pills label{margin:0;padding:6px 12px;border:1px solid var(--line);border-radius:999px;cursor:pointer;font-weight:500;background:var(--bg);transition:.15s}
.pills input:checked+label{background:var(--accent);border-color:var(--accent);color:#fff}
.pills input:focus-visible+label{outline:2px solid var(--accent);outline-offset:2px}
.side{position:sticky;top:18px;display:grid;gap:18px}
@media(max-width:900px){.side{position:static}}
.result{padding:24px;text-align:center}
.eyebrow{font-size:12px;letter-spacing:.12em;text-transform:uppercase;color:var(--muted);font-weight:600}
.gauge{position:relative;width:260px;max-width:100%;margin:6px auto 0}
.gauge svg{width:100%;display:block}
.track{stroke:var(--line)}
.bar{transition:stroke-dasharray .6s cubic-bezier(.2,.8,.2,1),stroke .3s}
.score{position:absolute;left:0;right:0;bottom:6px}
.score b{font:700 52px/1 Fraunces,Georgia,serif;font-variant-numeric:tabular-nums}
.score small{display:block;color:var(--muted);font-size:12px}
.badge{display:inline-block;margin-top:10px;padding:5px 14px;border-radius:999px;font-weight:600;font-size:13px;color:#fff;transition:background .3s}
.stats{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin-top:20px}
.stat{background:var(--bg);border:1px solid var(--line);border-radius:14px;padding:12px 6px}
.stat b{display:block;font:700 22px Fraunces,Georgia,serif;font-variant-numeric:tabular-nums}
.stat span{font-size:11.5px;color:var(--muted);line-height:1.3;display:block}
.sug{padding:20px 22px}
.sug h3{margin:0 0 10px;font:600 17px Fraunces,Georgia,serif}
.sug ol{margin:0;padding-left:20px}
.sug li{margin:7px 0}
.note{padding:14px 16px;border-radius:14px;background:var(--accent-soft);font-size:12.5px;color:var(--ink)}
.note b{color:var(--accent)}
.btn{width:100%;margin-top:18px;padding:12px;border:0;border-radius:12px;background:var(--accent);color:#fff;font:600 15px Inter,sans-serif;cursor:pointer}
.btn:hover{filter:brightness(1.07)}
.dim{opacity:.55;transition:opacity .2s}
.foot{margin-top:6px;color:var(--muted);font-size:12px;text-align:center}
</style>
</head>
<body>
<div class="wrap">
  <header>
    <div class="logo" aria-hidden="true">
      <svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M14 14.8V4a2 2 0 0 0-4 0v10.8a4 4 0 1 0 4 0z"/></svg>
    </div>
    <h1>TAAPSHAMAN</h1>
  </header>
  <p class="sub">Describe a building and its surroundings. The model estimates indoor heat, a preliminary Thermal Resilience Score, and screening ideas for retrofits. Results update as you adjust the inputs.</p>

  <div class="grid">
    <form id="form" onsubmit="return false"></form>

    <aside class="side">
      <div class="card result" id="result">
        <div class="eyebrow">Thermal Resilience Score</div>
        <div class="gauge">
          <svg viewBox="0 0 220 125" aria-hidden="true">
            <path class="track" d="M20 110A90 90 0 0 1 200 110" fill="none" stroke-width="16" stroke-linecap="round" pathLength="100"/>
            <path id="bar" class="bar" d="M20 110A90 90 0 0 1 200 110" fill="none" stroke="var(--mid)" stroke-width="16" stroke-linecap="round" pathLength="100" stroke-dasharray="0 100"/>
          </svg>
          <div class="score"><b id="score">–</b><small>out of 100</small></div>
        </div>
        <span class="badge" id="badge" style="background:var(--muted)">Calculating…</span>
        <div class="stats">
          <div class="stat"><b id="maxT">–</b><span>Daily max indoor °C</span></div>
          <div class="stat"><b id="hotT">–</b><span>Indoor °C on a hot day</span></div>
          <div class="stat"><b id="hrs">–</b><span>Hours above 28 °C</span></div>
        </div>
      </div>

      <div class="card sug">
        <h3>Potential retrofit measures</h3>
        <ol id="sug"></ol>
      </div>

      <div class="note"><b>Synthetic data.</b> The model was trained on synthetic outcomes, so predictions, the score and suggestions are illustrative only, not verified estimates for an actual building.
        <div id="metrics" style="margin-top:6px;color:var(--muted)"></div></div>
    </aside>
  </div>
</div>

<script>
const form = document.getElementById('form');
const state = {};
let timer, seq = 0;

const el = (tag, attrs = {}, ...kids) => {
  const n = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs)) k === 'class' ? n.className = v : n.setAttribute(k, v);
  kids.forEach(k => n.append(k));
  return n;
};
const slug = s => s.replace(/[^a-z0-9]+/gi, '-').toLowerCase();
const clean = s => s.replace(/\s*\((?:C, )?synthetic[^)]*\)/i, '').replace(/\s*\(synthetic\)/i, '').trim();
const unitOf = s => { const m = s.match(/\(([^)]*)\)/g); return m ? m.map(x => x.slice(1, -1)).join(' ').replace(/,?\s*synthetic,?\s*/i, '').replace(/^C$/, '°C').trim() : ''; };

function numberField(f) {
  const id = 'f-' + slug(f.name), unit = unitOf(f.name);
  const pad = (f.max - f.min) * 0.1 || 1;
  const lo = Math.max(0, Math.floor(f.min - pad)), hi = Math.ceil(f.max + pad);
  const out = el('span', { class: 'val' });
  const input = el('input', { type: 'range', id, min: lo, max: hi, step: f.step, value: f.default });
  const sync = () => { state[f.name] = parseFloat(input.value); out.textContent = state[f.name] + (unit && !unit.includes('.') ? ' ' + unit : ''); };
  input.addEventListener('input', () => { sync(); queue(); });
  sync();
  return el('div', { class: 'f' }, el('label', { for: id }, el('span', {}, clean(f.name)), out), input);
}

function choiceField(f) {
  state[f.name] = f.default;
  const wrap = el('div', { class: 'f' }, el('div', { class: 'lab' }, el('span', {}, clean(f.name))));
  if (f.choices.length <= 4) {
    const pills = el('div', { class: 'pills', role: 'radiogroup' });
    f.choices.forEach((c, i) => {
      const id = `f-${slug(f.name)}-${i}`;
      const r = el('input', { type: 'radio', name: slug(f.name), id, value: c });
      if (c === f.default) r.checked = true;
      r.addEventListener('change', () => { state[f.name] = c; queue(); });
      pills.append(r, el('label', { for: id }, c));
    });
    wrap.append(pills);
  } else {
    const sel = el('select', { 'aria-label': clean(f.name) });
    f.choices.forEach(c => sel.append(el('option', { value: c }, c)));
    sel.addEventListener('change', () => { state[f.name] = sel.value; queue(); });
    wrap.append(sel);
  }
  return wrap;
}

function build(schema) {
  schema.sections.forEach(sec => {
    const grid = el('div', { class: 'fields' });
    sec.fields.forEach(f => grid.append(f.type === 'number' ? numberField(f) : choiceField(f)));
    form.append(el('section', { class: 'card sec' }, el('h2', {}, sec.title), grid));
  });
  const btn = el('button', { class: 'btn', type: 'button' }, 'Assess this building');
  btn.addEventListener('click', predict);
  form.append(btn);
  const m = schema.metrics, k = Object.keys(m)[0];
  document.getElementById('metrics').textContent =
    `Trained on ${schema.n_rows} rows · held-out MAE ${m[k].mae} °C, R² ${m[k].r2} (synthetic).`;
}

const queue = () => { clearTimeout(timer); timer = setTimeout(predict, 250); };

async function predict() {
  const mine = ++seq;
  document.getElementById('result').classList.add('dim');
  try {
    const r = await fetch('/api/predict', { method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify(state) });
    const d = await r.json();
    if (mine !== seq) return;
    show(d);
  } catch (e) {
    document.getElementById('badge').textContent = 'Could not reach the model';
  }
  document.getElementById('result').classList.remove('dim');
}

function show(d) {
  const colour = d.score >= 75 ? 'var(--good)' : d.score >= 50 ? 'var(--mid)' : 'var(--bad)';
  const bar = document.getElementById('bar');
  bar.setAttribute('stroke-dasharray', `${d.score} 100`);
  bar.setAttribute('stroke', colour);
  document.getElementById('score').textContent = d.score.toFixed(1);
  const b = document.getElementById('badge');
  b.textContent = d.rating; b.style.background = colour;
  document.getElementById('maxT').textContent = d.max_temp.toFixed(1);
  document.getElementById('hotT').textContent = d.hot_temp.toFixed(1);
  document.getElementById('hrs').textContent = d.hours_above_28.toFixed(1);
  const ol = document.getElementById('sug'); ol.replaceChildren();
  d.suggestions.forEach(s => ol.append(el('li', {}, s)));
}

fetch('/api/schema').then(r => { if (!r.ok) throw new Error(r.status); return r.json(); })
  .then(s => { build(s); predict(); })
  .catch(() => {
    form.append(el('div', { class: 'card sec' },
      el('h2', {}, 'Cannot reach the model'),
      el('p', {}, 'Start the server with "python app.py" and open http://127.0.0.1:5000 in your browser. Opening index.html directly from a folder will not work, and the terminal will show any error (for example a missing CSV).')));
    document.getElementById('badge').textContent = 'Model not running';
  });
</script>
</body>
</html>
'''

# ------------------------------------------------------------ 4. server
app = Flask(__name__)

@app.get("/")
def index():
    return Response(HTML, mimetype="text/html")

@app.get("/api/schema")
def schema():
    return jsonify(SCHEMA)

@app.post("/api/predict")
def predict():
    data = request.get_json(force=True)
    row = {c: (float(data.get(c)) if c in numeric else str(data.get(c))) for c in features}
    frame = pd.DataFrame([row], columns=features)
    max_t, hot_t, hours = (float(MODELS[t].predict(frame)[0]) for t in TARGETS)
    score = float(np.clip(100 * (40 - max_t) / 14, 0, 100))
    rating = ("Higher preliminary resilience" if score >= 75 else
              "Moderate preliminary resilience" if score >= 50 else
              "Lower preliminary resilience")
    return jsonify({
        "max_temp": round(max_t, 1), "hot_temp": round(hot_t, 1),
        "hours_above_28": round(float(np.clip(hours, 0, 24)), 1),
        "score": round(score, 1), "rating": rating, "suggestions": suggest(row),
    })

# Stop a previous run of this cell (if any) so the port is free, then start.
try:
    _srv.shutdown()
except Exception:
    pass
_srv = make_server("0.0.0.0", PORT, app, threaded=True)
Thread(target=_srv.serve_forever, daemon=True).start()
print("Ready. Showing the app below.")
output.serve_kernel_port_as_iframe(PORT, height=950)
